# Character CNN (Model A) on DHCD - Google Colab

Owner: Ravi. Whole notebook: about 15-25 minutes on a T4.

Before you start:
1. **Runtime menu -> Change runtime type -> Hardware accelerator = T4 GPU -> Save.**
2. **Left sidebar -> key icon (Secrets) -> Add new secret:** name `GITHUB_TOKEN`, value = a GitHub token that can read the repo, and switch **Notebook access** on.
   Nikhil's old `kaggle-read` token expired after 7 days. Easiest: make your own **classic** token: GitHub -> Settings -> Developer settings -> Personal access tokens -> **Tokens (classic)** -> Generate new token (classic) -> tick **repo** -> Expiration 7 days -> Generate, copy it. It works because you are a collaborator on the repo. If you add no secret, cell 2 asks you to paste the token instead.
3. Run the cells **one at a time** (Shift+Enter). Keep this browser tab open while a cell runs.
4. At the end, cell 8 downloads `char_cnn_outputs.zip`. Put it in your `Downloads` folder.

In [ ]:
# 1. GPU check: must print True and a GPU name
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU -> Runtime > Change runtime type > T4 GPU")

In [ ]:
# 2. get the code (private repo). The char CNN code is on main.
# Token order: Colab secret GITHUB_TOKEN (browser Colab) -> otherwise you are asked to paste it.
def read_token():
    try:
        from google.colab import userdata
        t = userdata.get("GITHUB_TOKEN")  # only works in the Colab browser UI, not in VS Code
        if t:
            return t
    except Exception as e:
        print("no Colab secret here (" + type(e).__name__ + "): paste the token instead")
    from getpass import getpass
    t = getpass("Paste your GitHub token (input is hidden): ")
    if isinstance(t, dict):  # the VS Code notebook UI returns a dict instead of a string
        t = t.get("value") or next(iter(t.values()), "")
    return str(t).strip()

token = read_token()
assert token.startswith(("ghp_", "github_pat_")), "that does not look like a GitHub token"
!git clone -q https://{token}@github.com/Nikhil04-ai/Devanagari_Word_Recognition_Project.git
%cd /content/Devanagari_Word_Recognition_Project
!git remote set-url origin https://github.com/Nikhil04-ai/Devanagari_Word_Recognition_Project.git
!git checkout -q main
!git log --oneline -3
!pip install -q rapidfuzz pytest pyyaml
del token

In [ ]:
# 3. data: DHCD from the UCI repository (about 80 MB). The code reads the zip directly, no unzip needed.
!mkdir -p data/raw/dhcd
!wget -q -O data/raw/dhcd/dhcd.zip "https://archive.ics.uci.edu/static/public/389/devanagari+handwritten+character+dataset.zip"
!ls -lh data/raw/dhcd   # expect dhcd.zip, about 77M

In [ ]:
# 4. unit tests for the char CNN: expect "7 passed"
!python -m pytest -q tests/test_char_cnn.py

In [ ]:
# 5. sanity check: memorise 100 training images. Must end with "overfit test passed"
!python -m src.char_cnn.train --config configs/char_cnn/overfit.yaml

In [ ]:
# 6. full training: 15 epochs (well under 1 minute each on a T4). Expect val acc around 98-99 %.
# If Colab disconnects: run cells 1-3 again, then this line with --resume added.
!python -m src.char_cnn.train --config configs/char_cnn/full.yaml

In [ ]:
# 7. evaluate ONCE on the test set -> results/char_cnn_full_argmax/
!python -m src.char_cnn.evaluate --ckpt checkpoints/char_cnn_full/best.pt
!cat results/char_cnn_full_argmax/metrics.json

In [ ]:
# 8. pack results + training log + best checkpoint and download them
!cp checkpoints/char_cnn_full/log.csv results/char_cnn_full_argmax/training_log.csv
!zip -q -r /content/char_cnn_outputs.zip results/char_cnn_full_argmax checkpoints/char_cnn_full/best.pt
!ls -lh /content/char_cnn_outputs.zip
from google.colab import files
files.download("/content/char_cnn_outputs.zip")